# 01 — Exploring the OpenABC-D dataset

Goal (PRD §6.2): understand the graph files, recipes and labels **before** writing any model code.
Findings are summarised in `docs/data_notes.md`.

Run the cells top to bottom. Set `DATA_ROOT` below to wherever the dataset lives on your machine.

In [ ]:
import pickle
import sys
from pathlib import Path

import pandas as pd
import torch

# Make "src" importable when the notebook runs from the notebooks/ folder
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
from src.data import load_aig_file

# The only setting: where the dataset is (read-only; we never write here)
DATA_ROOT = REPO_ROOT / "OPENABC2_DATASET"
PROCESSED = DATA_ROOT / "processed"
print("Dataset found:", PROCESSED.exists())

## 1. One graph file

Each `<design>_syn<N>_step0.pt.zip` holds the **starting** circuit (step 0, before synthesis) as an
And-Inverter Graph, plus recipe number N. The files were saved with an old PyTorch Geometric version,
so `load_aig_file` rebuilds them in the current format (see `src/data.py`).

In [ ]:
g = load_aig_file(PROCESSED / "ss_pcm_syn0_step0.pt.zip")
print(g)
for key, value in g.items():
    if torch.is_tensor(value):
        print(f"{key:28s} tensor {tuple(value.shape)}  first values: {value.flatten()[:8].tolist()}")
    else:
        print(f"{key:28s} {type(value).__name__}: {str(value)[:60]}")

## 2. What the node and edge features mean

- `node_type`: 0 = primary input (PI), 1 = primary output (PO), 2 = AND gate
- `edge_type`: 1 = inverted wire (a NOT on the wire), 0 = normal wire
- `num_inverted_predecessors`: how many of a node's incoming wires are inverted (0, 1 or 2)

**Edge direction:** `edge_index[0]` is the gate *receiving* the signal, `edge_index[1]` is the source.
Check: in an AIG every AND gate has exactly 2 inputs, outputs have 1, and inputs have 0.

*Key point:* PyG sends messages from `edge_index[0]` to `edge_index[1]` by default, so as stored,
information would flow from outputs back towards inputs. The model must flip or symmetrise the edges.

In [ ]:
fan_in = torch.bincount(g.edge_index[0], minlength=g.num_nodes)
for t, name in [(0, "PI"), (1, "PO"), (2, "AND")]:
    print(f"{name:4s}: {int((g.node_type == t).sum()):4d} nodes, incoming wires per node = {sorted(set(fan_in[g.node_type == t].tolist()))}")

inverted_in = torch.bincount(g.edge_index[0][g.edge_type == 1], minlength=g.num_nodes)
print("num_inverted_predecessors == count of inverted incoming wires:", torch.equal(inverted_in, g.num_inverted_predecessors))

## 3. Is the same graph stored once per recipe?

If the circuit is identical in all 1500 files of a design, we only need to load it once per design
(saves memory and time in M3).

In [ ]:
GRAPH_KEYS = ["edge_index", "node_type", "num_inverted_predecessors", "edge_type"]
g0 = load_aig_file(PROCESSED / "ss_pcm_syn0_step0.pt.zip")
for n in [1, 777, 1499]:
    gn = load_aig_file(PROCESSED / f"ss_pcm_syn{n}_step0.pt.zip")
    same_graph = all(torch.equal(g0[k], gn[k]) for k in GRAPH_KEYS)
    print(f"syn{n}: same graph = {same_graph}, same recipe = {torch.equal(g0.synVec, gn.synVec)}")

## 4. Recipes

`synVec` is the recipe: 20 integers, one per ABC command, in order.
Mapping from OpenABC `datagen/utilities/synthID2SeqMapping.py`. Recipe N is the same for every design.

In [ ]:
ABC_COMMANDS = {0: "refactor", 1: "refactor -z", 2: "rewrite", 3: "rewrite -z",
                4: "resub", 5: "resub -z", 6: "balance"}

recipe = g0.synVec.tolist()
print("Recipe 0 as numbers:", recipe)
print("Recipe 0 as ABC commands:")
print("  " + "; ".join(ABC_COMMANDS[c] for c in recipe))

other = load_aig_file(PROCESSED / "aes_syn0_step0.pt.zip").synVec
print("\nRecipe 0 identical for ss_pcm and aes:", torch.equal(g0.synVec, other))

## 5. Labels

`synthesisStatistics.pickle`: for each design, 5 lists of 1500 values (one per recipe, indexed by synID).
Column order from OpenABC `datagen/utilities/pickleStatsForML.py`. The pickle holds only plain Python
data (no classes), so loading it cannot run code.

In [ ]:
with open(DATA_ROOT / "synthesisStatistics.pickle", "rb") as f:
    stats = pickle.load(f)

LABEL_NAMES = ["final_AND", "final_NOT", "final_levels", "area", "delay"]
labels = pd.DataFrame(dict(zip(LABEL_NAMES, stats["ss_pcm"])))
labels.index.name = "synID"
print("ss_pcm, initial AND gates:", int(g0.and_nodes))
labels.describe().round(2)

Note how few distinct values `final_AND` has for a small design. That limits how well any model can
*rank* recipes by AND count. The mapped `area` (column 3) varies much more.

In [ ]:
labels.nunique()

## 6. Size of every design

Loads recipe 0 of each design (graphs are identical across recipes). `origDict` is the value in
`origAIGNodesDict.pickle`, compared with the graph's own AND count. This takes a minute.

In [ ]:
with open(DATA_ROOT / "origAIGNodesDict.pickle", "rb") as f:
    orig = pickle.load(f)

rows = []
for des in sorted(stats):
    gd = load_aig_file(PROCESSED / f"{des}_syn0_step0.pt.zip")
    assert gd.edge_index.min() >= 0 and gd.edge_index.max() < gd.num_nodes, f"{des}: bad edge index"
    rows.append({"design": des, "nodes": gd.num_nodes, "edges": gd.edge_index.shape[1],
                 "AND": int(gd.and_nodes), "PI": int(gd.pi), "PO": int(gd.po),
                 "depth": int(gd.longest_path), "origDict": orig.get(des),
                 "final_AND_min": min(stats[des][0]), "final_AND_max": max(stats[des][0])})
sizes = pd.DataFrame(rows).set_index("design").sort_values("nodes")
sizes

In [ ]:
matches = sizes.dropna(subset=["origDict"])
print("origAIGNodesDict == initial AND count for", int((matches["origDict"] == matches["AND"]).sum()), "of", len(matches), "designs")
print("Mismatches:")
print(matches.loc[matches["origDict"] != matches["AND"], ["AND", "origDict"]])
print("In origAIGNodesDict but not in the dataset:", sorted(set(orig) - set(stats)))

## 7. Splits provided by OpenABC

Each CSV lists graph file names. If the train and test files share designs, it is a split by recipe;
if not, it is a split by design (unseen circuits).

In [ ]:
def designs_in(csv_name):
    names = pd.read_csv(DATA_ROOT / csv_name)["fileName"]
    return set(names.str.replace(r"_syn\d+_step\d+\.pt\.zip$", "", regex=True)), len(names)

for split in ["set1", "set2", "mixmatchv1", "ncsu_set1"]:
    train, n_train = designs_in(f"train_data_{split}.csv")
    test, n_test = designs_in(f"test_data_{split}.csv")
    kind = "by recipe" if train & test else "by design"
    print(f"{split:11s} train {n_train:5d} files / {len(train)} designs | test {n_test:5d} files / {len(test)} designs | shared {len(train & test):2d} -> {kind}")

## 8. Data quality

In [ ]:
import math

bad = {des: sum(1 for col in cols for x in col if (isinstance(x, float) and math.isnan(x)) or x < 0)
       for des, cols in stats.items()}
print("Designs with NaN or negative labels:", {d: n for d, n in bad.items() if n})
print("Every label list has length 1500:", all(len(c) == 1500 for cols in stats.values() for c in cols))
print("Designs whose final AND count never changes:", [d for d, cols in stats.items() if len(set(cols[0])) == 1])